In [9]:
# ppo_fetch_dense_plots.py
# ------------------------------------------------------------
# PPO on FetchPickAndPlaceDense-v4 (MuJoCo) with randomized starts,
# GPU support, VecNormalize (train & eval), and plots.
# Outputs go into ./runs_fetch_ppo/
# ------------------------------------------------------------

import os, glob, math
os.environ.setdefault("MUJOCO_GL", "glfw")  # GUI backend on Windows

# Keep matplotlib headless on Windows/servers
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import gymnasium as gym
import gymnasium_robotics
from gymnasium.wrappers import FlattenObservation

import numpy as np
import torch

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback

# For reading TensorBoard event files robustly
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# =========================
# Config
# =========================
LOG_DIR = "./runs_fetch_ppo"
TB_DIR  = os.path.join(LOG_DIR, "tb")  # SB3 writes here
os.makedirs(LOG_DIR, exist_ok=True)

NUM_ENVS = 4             # bump if your CPU can handle more
TOTAL_STEPS = 1_000_000    # train longer (1–3M) for real learning
OBJ_RANGE = 0.15         # spawn randomization for block
TARGET_RANGE = 0.20      # spawn randomization for goal
EVAL_EPISODES = 10

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Register robotics envs once
gym.register_envs(gymnasium_robotics)

# =========================
# Env factories
# =========================
def make_train_env(seed=0, idx=0, obj_range=OBJ_RANGE, target_range=TARGET_RANGE):
    """
    Training env:
    - FetchPickAndPlaceDense-v4 (MuJoCo)
    - randomized obj/goal ranges
    - FlattenObservation (Dict -> Box)
    - Monitor (per-env CSV)
    """
    def _thunk():
        env = gym.make("FetchPickAndPlaceDense-v4")
        env.unwrapped.obj_range = obj_range
        env.unwrapped.target_range = target_range
        env = FlattenObservation(env)
        mon_file = os.path.join(LOG_DIR, f"monitor_env{idx}.csv")
        env = Monitor(env, mon_file)
        env.reset(seed=seed)
        return env
    return _thunk

def make_eval_env(seed=123, obj_range=OBJ_RANGE, target_range=TARGET_RANGE):
    """
    Raw eval env with same wrappers as train (we’ll wrap in VecNormalize with saved stats).
    """
    env = gym.make("FetchPickAndPlaceDense-v4")
    env.unwrapped.obj_range = obj_range
    env.unwrapped.target_range = target_range
    env = FlattenObservation(env)
    env = Monitor(env)
    env.reset(seed=seed)
    return env

# =========================
# Minimal callback (future-proof)
# =========================
class KeepAliveCallback(BaseCallback):
    """No-op callback only to satisfy BaseCallback interface on all SB3 versions."""
    def _on_step(self) -> bool:
        return True

cb = KeepAliveCallback()

# =========================
# Build vectorized training env
# =========================
train_env = DummyVecEnv([make_train_env(seed=i, idx=i) for i in range(NUM_ENVS)])
train_env = VecNormalize(
    train_env, training=True,
    norm_obs=True, norm_reward=True,
    clip_obs=10.0, clip_reward=10.0
)

# =========================
# PPO model (GPU if available)
# =========================
model = PPO(
    "MlpPolicy",
    train_env,
    verbose=1,
    n_steps=2048,
    batch_size=32,
    n_epochs=6,
    gamma=0.98,
    gae_lambda=0.95,
    learning_rate=3e-4,
    clip_range=0.25,
    ent_coef=0.05,
    vf_coef=0.25,
    tensorboard_log=TB_DIR,
    seed=0,
    device=DEVICE,
)

# (Optional) small extra speed on PyTorch 2.x
if torch.cuda.is_available():
    try:
        torch.set_float32_matmul_precision("high")
        model.policy = torch.compile(model.policy)  # safe to skip if it errors
        print("Enabled torch.compile for policy.")
    except Exception as e:
        print(f"torch.compile not enabled: {e}")

# =========================
# Train
# =========================
model.learn(total_timesteps=TOTAL_STEPS, callback=cb)

# Save model + normalization stats
model_path = os.path.join(LOG_DIR, "ppo_fetch_dense.zip")
stats_path = os.path.join(LOG_DIR, "vecnorm.pkl")
model.save(model_path)
train_env.save(stats_path)

# Freeze stats for any later use
train_env.training = False
train_env.norm_reward = False

# =========================
# Evaluation (with SAME VecNormalize stats)
# =========================
def eval_success_and_returns(episodes=EVAL_EPISODES):
    raw_eval_env = make_eval_env()
    vec_eval_env = DummyVecEnv([lambda: raw_eval_env])
    vec_eval_env = VecNormalize.load(stats_path, vec_eval_env)
    vec_eval_env.training = False
    vec_eval_env.norm_reward = False

    ep_successes, ep_returns = [], []
    for _ in range(episodes):
        obs = vec_eval_env.reset()
        done = np.array([False])
        ret = 0.0
        ep_success = 0.0

        while not done[0]:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, infos = vec_eval_env.step(action)
            ret += float(reward[0])
            done = np.logical_or(terminated, truncated)
            info0 = infos[0] if isinstance(infos, (list, tuple)) else infos
            if isinstance(info0, dict) and "is_success" in info0:
                ep_success = max(ep_success, float(info0["is_success"]))

        ep_successes.append(ep_success)
        ep_returns.append(ret)

    vec_eval_env.close()
    return np.array(ep_successes), np.array(ep_returns)

sr, rets = eval_success_and_returns()
print(f"Eval success rate over {EVAL_EPISODES} episodes: {sr.mean():.2f}")
print(f"Eval mean return: {rets.mean():.2f} ± {rets.std():.2f}")

# =========================
# Plot helpers
# =========================
def load_monitor_returns(pattern):
    """
    Load episodic returns from gym Monitor CSVs.
    Skips any lines starting with '#'.
    """
    paths = sorted(glob.glob(pattern))
    returns, lengths = [], []
    for p in paths:
        with open(p, "r") as f:
            rows = [ln.strip() for ln in f if ln and not ln.startswith("#")]
        if not rows:
            continue
        data = np.genfromtxt(rows, delimiter=",")
        if data is None or (isinstance(data, float) and np.isnan(data)):
            continue
        if data.ndim == 1 and data.size == 3:
            data = data.reshape(1, 3)
        if data.size:
            # columns: r, l, t
            returns.extend(data[:, 0].tolist())
            lengths.extend(data[:, 1].tolist())
    return np.array(returns), np.array(lengths)

def smooth(x, k=20):
    if len(x) < 2:
        return x
    k = max(1, min(k, len(x)//5))
    w = np.ones(k) / k
    return np.convolve(x, w, mode="valid")

# =========================
# Make plots
# =========================
# 1) Training episodic return (smoothed)
train_returns, train_lengths = load_monitor_returns(os.path.join(LOG_DIR, "monitor_env*.csv"))
if train_returns.size:
    plt.figure()
    plt.plot(smooth(train_returns, k=20))
    plt.title("Training Episodic Return (smoothed)")
    plt.xlabel("Episode")
    plt.ylabel("Return")
    plt.tight_layout()
    plt.savefig(os.path.join(LOG_DIR, "training_reward.png"), dpi=150)
    plt.close()

# 2) Eval success per episode
plt.figure()
plt.bar(range(len(sr)), sr)
plt.title(f"Eval Success per Episode (mean={sr.mean():.2f})")
plt.xlabel("Episode")
plt.ylabel("Success (0/1)")
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, "success_rate.png"), dpi=150)
plt.close()

# 3) PPO losses / metrics from TensorBoard events (SB3-version-agnostic)
def find_tb_event_files(tb_root):
    # SB3 usually writes under tb/{algo}_{runid}/events.*
    if not os.path.isdir(tb_root):
        return []
    event_files = []
    for root, _, files in os.walk(tb_root):
        for f in files:
            if f.startswith("events."):
                event_files.append(os.path.join(root, f))
    return sorted(event_files)

def load_tb_scalars(event_paths, tags):
    """Return dict[tag] -> (steps, values) aggregated over all event files."""
    out = {t: ([], []) for t in tags}
    for p in event_paths:
        try:
            ea = EventAccumulator(p)
            ea.Reload()
            for t in tags:
                if t in ea.Tags().get("scalars", []):
                    sc = ea.Scalars(t)
                    steps = [s.step for s in sc]
                    vals  = [s.value for s in sc]
                    out[t][0].extend(steps)
                    out[t][1].extend(vals)
        except Exception:
            pass
    # sort by step
    for t in tags:
        steps, vals = out[t]
        if steps:
            idx = np.argsort(steps)
            out[t] = (list(np.array(steps)[idx]), list(np.array(vals)[idx]))
    return out

tb_files = find_tb_event_files(TB_DIR)
tb_tags  = [
    "train/value_loss",
    "train/policy_gradient_loss",
    "train/entropy_loss",
    "train/approx_kl",
    "rollout/ep_rew_mean",
    "train/clip_fraction",
]
scalars = load_tb_scalars(tb_files, tb_tags)

plotted = False
plt.figure()
for t in tb_tags:
    steps, vals = scalars[t]
    if steps:
        plt.plot(steps, vals, label=t)
        plotted = True
if plotted:
    plt.title("PPO Training Losses / Metrics (from TensorBoard)")
    plt.xlabel("Timesteps")
    plt.ylabel("Value")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(LOG_DIR, "ppo_losses.png"), dpi=150)
plt.close()

print(f"\nSaved model & plots in: {os.path.abspath(LOG_DIR)}")

# =========================
# Optional: quick post-training visual rollout (comment out if not needed)
# =========================
# import time
# vis_env = gym.make("FetchPickAndPlaceDense-v4", render_mode="human")
# vis_env.unwrapped.obj_range = OBJ_RANGE
# vis_env.unwrapped.target_range = TARGET_RANGE
# vis_env = FlattenObservation(vis_env)
# obs, _ = vis_env.reset(seed=0)
# for _ in range(300):
#     action, _ = model.predict(obs, deterministic=True)
#     obs, _, term, trunc, _ = vis_env.step(action)
#     vis_env.render()
#     if term or trunc:
#         obs, _ = vis_env.reset()
#         time.sleep(0.02)
# vis_env.close()


Using device: cpu
Using cpu device
Logging to ./runs_fetch_ppo\tb\PPO_3
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -13      |
|    success_rate    | 0.04     |
| time/              |          |
|    fps             | 443      |
|    iterations      | 1        |
|    time_elapsed    | 18       |
|    total_timesteps | 8192     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -11.5       |
|    success_rate         | 0.03        |
| time/                   |             |
|    fps                  | 270         |
|    iterations           | 2           |
|    time_elapsed         | 60          |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.019421566 |
|    clip_fraction        | 0.102       |
|    cli

KeyboardInterrupt: 